# Imports

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/ncdc-lassa-fever-timeseries-20202025/lassa_fever_timeseries_minimal.csv
/kaggle/input/ncdc-lassa-fever-timeseries-20202025/lassa_fever_timeseries_full.csv


# Functions

In [2]:
# Mean Absolute Error (MAE)
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

# Root Mean Squared Error (RMSE)
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

# Mean Absolute Percentage Error (MAPE)
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0  # avoid division by zero
    return (np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

# Function to calculate all three metrics
def calculate_evaluation_metrics(y_true, y_pred):
    metrics = []
    metrics.append(mae(y_true, y_pred).round(2))
    metrics.append(rmse(y_true, y_pred).round(2))
    metrics.append(mape(y_true, y_pred).round(2))
    return metrics


metrics_table = pd.DataFrame(columns=["Model", "MAE", "RMSE", "MAPE"])
metrics_table.set_index('Model')

,MAE,RMSE,MAPE
Model,,,


# Data and EDA

In [3]:
# read data into a dataframe
df = pd.read_csv('/kaggle/input/ncdc-lassa-fever-timeseries-20202025/lassa_fever_timeseries_minimal.csv')

# construct series dataframe
ts_data = df[['week_end_date', 'epi_week', 'confirmed_cases']].copy()
ts_data.columns = ['date', 'epi_week', 'confirmed_cases']

# Ensure datetime index
ts_data['date'] = pd.to_datetime(ts_data['date'])
ts_data = ts_data.set_index('date')
ts_data = ts_data.asfreq('W') # set frequency to weekly
ts_data

,epi_week,confirmed_cases
date,,
2020-01-05,1,18
2020-01-12,2,64
2020-01-19,3,81
2020-01-26,4,95
2020-02-02,5,104
...,...,...
2025-10-19,42,9
2025-10-26,43,11
2025-11-02,44,12


In [4]:
# Some basic EDA
print(ts_data.dtypes)
print(ts_data.isnull().sum())
print(ts_data.describe())

epi_week           int64
confirmed_cases    int64
dtype: object
epi_week           0
confirmed_cases    0
dtype: int64
         epi_week  confirmed_cases
count  307.000000       307.000000
mean    26.136808        20.540717
std     14.879500        25.897753
min      1.000000         0.000000
25%     13.000000         6.000000
50%     26.000000        10.000000
75%     39.000000        21.000000
max     53.000000       137.000000


In [5]:
# create and display plot 
fig =  px.line(ts_data, x=ts_data.index, y="confirmed_cases",
        title="Weekly Lassa Fever Cases (2020–2025)")
fig.show()

In [6]:
# Functions 
def plot_func(forecast, title):
    """Function to plot the forecasts"""
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=train.index, y=train['confirmed_cases'], name="Train"))
    fig.add_trace(go.Scatter(x=test.index, y=forecast, name="Test"))
    fig.update_layout(template="simple_white", font=dict(size=18), title_text=title,
                     width=650, title_x=0.5, height=400, xaxis_title='Date',
                     yaxis_title='Confirmed Cases')
    return fig.show()

# Train Test Split

In [7]:
# Train Test Split
train = ts_data.iloc[:-int(len(ts_data)*0.2)].copy()
test = ts_data.iloc[-int(len(ts_data)*0.2):].copy()

# Naive Forcast

In [8]:
# Naive Forcast
test['naive_forecast'] = train['confirmed_cases'].iloc[-1]
plot_func(test['naive_forecast'], "Naive Forecast")

# Calculate metrics and add to results dataframe
metrics_list = ['Naive Forecast'] + calculate_evaluation_metrics(test['confirmed_cases'], test['naive_forecast'])
metrics_table.loc[len(metrics_table)] = metrics_list

# Average Forecast

In [9]:
# Average Forecast
test['mean_forecast'] = train['confirmed_cases'].mean()
plot_func(test['mean_forecast'], 'Average Forecast')

# Calculate metrics and add to results dataframe
metrics_list = ['Average Forecast'] + calculate_evaluation_metrics(test['confirmed_cases'], test['mean_forecast'])
metrics_table.loc[len(metrics_table)] = metrics_list

# Drift Forecast

In [10]:
# Drift forecast
constant = (train['confirmed_cases'].iloc[-1] - train['confirmed_cases'].iloc[0])/(len(train)-1)
test['h'] = range(len(test))
test['drift_forecast'] = train['confirmed_cases'].iloc[-1] + test['h']*constant

plot_func(test['drift_forecast'], 'Drift Forecast')

# Calculate metrics and add to results dataframe
metrics_list = ['Drift Forecast'] + calculate_evaluation_metrics(test['confirmed_cases'], test['drift_forecast'])
metrics_table.loc[len(metrics_table)] = metrics_list

# Seasonal Naive Forecast

In [11]:
# Seasonal Naive Forecast

snaive_fc = []
for index, row in test.iterrows():

    # get confrimed cases of the curresponding week last year 
    lag_52 = train.loc[train['epi_week']==row['epi_week']].iloc[-1] # however this fails when dealing with 53 weeks
    snaive_fc.append(lag_52['confirmed_cases'])

plot_func(snaive_fc, 'Seasonal Naive Forecast')

# Calculate metrics and add to results dataframe
metrics_list = ['Seasonal Naive Forecast'] + calculate_evaluation_metrics(test['confirmed_cases'], snaive_fc)
metrics_table.loc[len(metrics_table)] = metrics_list

In [12]:
metrics_table

,Model,MAE,RMSE,MAPE
0,Naive Forecast,16.33,26.37,60.30
1,Average Forecast,16.20,20.94,129.03
2,Drift Forecast,17.69,27.12,69.84
3,Seasonal Naive Forecast,8.74,15.53,49.24
